In [ ]:
from pathlib import Path
import sys
import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw, ImageFont

import ipywidgets as widgets
from IPython.display import display, clear_output

PROJECT_ROOT = Path("..").resolve()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import visual_identity_solver
import visual_cleaning

importlib.reload(visual_identity_solver)
importlib.reload(visual_cleaning)

solver = visual_identity_solver

from visual_cleaning import clean_visual_dataframe, add_clean_emotion_columns

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

DEBATE_NAME = "Ventura_vs_Seguro_November_17"

print("Project root:", PROJECT_ROOT)
print("Debate:", DEBATE_NAME)
print("Loaded solver from:", visual_identity_solver.__file__)
print("Loaded cleaning from:", visual_cleaning.__file__)

In [ ]:
visual_candidates = list(DATA_DIR.rglob(f"*{DEBATE_NAME}*visual*.pkl"))

if len(visual_candidates) == 0:
    print("Available visual pickle files:")
    for path in DATA_DIR.rglob("*visual*.pkl"):
        print(path)

    raise FileNotFoundError(f"No visual pickle found for {DEBATE_NAME}")

VISUAL_PKL = visual_candidates[0]

print("Using visual pickle:")
print(VISUAL_PKL)

In [ ]:
solver_output_path = PROCESSED_DIR / f"{DEBATE_NAME}_identity_solver_SINGLE_UNCONSTRAINED_CLEANED_predictions.pkl"
solver_frames_path = PROCESSED_DIR / f"{DEBATE_NAME}_identity_solver_SINGLE_UNCONSTRAINED_CLEANED_frames.pkl"

FORCE_RERUN = True

# Since you installed the libraries, try True.
# If InsightFace fails or coverage is low, the solver falls back automatically.
USE_INSIGHTFACE = True

if solver_output_path.exists() and solver_frames_path.exists() and not FORCE_RERUN:
    print("Loading cached single-unconstrained solver results...")
    out = pd.read_pickle(solver_output_path)
    df_solver = pd.read_pickle(solver_frames_path)
    model_name = "cached_single_unconstrained"

else:
    cfg = solver.Config(
        pkl=VISUAL_PKL,
        project_root=PROJECT_ROOT,
        frames_root=Path("Frames"),
        out=PROCESSED_DIR / f"{DEBATE_NAME}_identity_solver_single_unconstrained_outputs",

        use_insightface=USE_INSIGHTFACE,

        # Multi-person constraints stay active
        force_small_two_has_person2=True,

        # IMPORTANT:
        # Do not use early single shots as person_2 anchors.
        # Single-person shots should be model/prototype predictions only.
        use_first_single_p2_prior=False,
        early_single_p2_always=False,

        constraint_first=True,

        prototype_temperature=2.25,
        low_confidence_threshold=0.55,

        two_large_sum_area=0.55,
        two_touching_gap=0.035,

        save_debug_images=False,
        annotate_every=0,
    )

    print("Loading raw pickle...")
    df_raw = solver.load_visual_pickle(VISUAL_PKL)

    print("Cleaning visual detections first...")
    df_clean = clean_visual_dataframe(df_raw)
    df_clean = add_clean_emotion_columns(df_clean)

    # IMPORTANT:
    # The solver expects columns named Poses and Fer.
    # We replace them with the cleaned detections.
    df_solver = df_clean.copy()
    df_solver["Poses"] = df_solver["Clean_Poses"]
    df_solver["Fer"] = df_solver["Clean_Fer"]

    df_solver["frame_num"] = df_solver["Frame"].map(solver.parse_frame_num)
    df_solver = df_solver.sort_values("frame_num").reset_index(drop=True)

    df_solver["n_poses"] = df_solver["Poses"].map(
        lambda x: len(x) if isinstance(x, list) else 0
    )

    df_solver["n_faces"] = df_solver["Fer"].map(
        lambda x: sum(1 for f in x if isinstance(f, dict)) if isinstance(x, list) else 0
    )

    print("Cleaned frame composition:")
    display(pd.crosstab(df_solver["n_poses"], df_solver["n_faces"]))

    print("Inferring frame size...")
    width, height = solver.infer_frame_size(df_solver, PROJECT_ROOT)
    print("Frame size:", width, height)

    print("Building detection table from CLEANED boxes...")
    det = solver.build_detection_table(df_solver, cfg, width, height)
    print("Cleaned detections:", len(det))

    print("Creating base features...")
    base_X, base_names, aux = solver.create_base_features(det, width, height)
    det = pd.concat([det.reset_index(drop=True), aux.reset_index(drop=True)], axis=1)

    print("Extracting InsightFace embeddings if available...")
    arc_X = solver.extract_insightface_embeddings(det, cfg)

    print("Building model features...")
    X = solver.build_model_features(det, base_X, arc_X, cfg)

    print("Assigning weak labels...")
    det = solver.assign_weak_labels(det, cfg)

    print("Weak anchor counts:")
    display(
        det.loc[det["weak_label"] >= 0, "weak_label"]
        .map(solver.INT_TO_LABEL)
        .value_counts()
    )

    print("Fitting identity model...")
    if cfg.constraint_first:
        probs, pred, model_name = solver.fit_prototype_identity_model(X, det, cfg)
    else:
        probs, pred, model_name = solver.fit_identity_model(X, det, cfg)

    print("Model:", model_name)

    print("Applying frame constraints...")
    out = solver.apply_frame_constraints(det, probs, cfg)

    # ------------------------------------------------------------
    # IMPORTANT CORRECTION:
    # Do NOT smooth single-person runs.
    # Do NOT apply final frame constraints to single-person shots.
    # Single-person shots should stay as raw model/prototype predictions.
    # ------------------------------------------------------------

    out["unconstrained_model_person"] = out["model_person"]
    out["unconstrained_model_confidence"] = out["model_confidence"]

    single_mask = out["n_faces"] == 1

    out.loc[single_mask, "person_label"] = out.loc[single_mask, "unconstrained_model_person"]
    out.loc[single_mask, "confidence"] = out.loc[single_mask, "unconstrained_model_confidence"]
    out.loc[single_mask, "assignment_source"] = "model_single_unconstrained"

    # For widget display, make model_person match the final chosen label.
    # The original model output is preserved in unconstrained_model_person.
    out["model_person"] = out["person_label"]
    out["model_confidence"] = out["confidence"]

    out.to_pickle(solver_output_path)
    df_solver.to_pickle(solver_frames_path)

    print("Saved:")
    print(solver_output_path)
    print(solver_frames_path)

print("Final assignments:")
display(out["person_label"].value_counts())

print("Assignment sources:")
display(out["assignment_source"].value_counts())

print("Person 2 detections by source:")
display(
    out[out["person_label"] == "person_2"]
    ["assignment_source"]
    .value_counts()
)

In [ ]:
def resolve_frame_path_for_widget(frame_value):
    path = solver.resolve_frame_path(
        PROJECT_ROOT,
        Path("Frames"),
        frame_value,
    )

    if path is not None and path.exists():
        return path

    return None


def get_font(size=18):
    try:
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except:
        return ImageFont.load_default()


def draw_text(draw, x, y, text, fill=(255, 255, 0), font_size=18):
    font = get_font(font_size)
    box = draw.textbbox((x, y), text, font=font)

    draw.rectangle(
        [box[0] - 4, box[1] - 4, box[2] + 4, box[3] + 4],
        fill=(0, 0, 0),
    )

    draw.text(
        (x, y),
        text,
        fill=fill,
        font=font,
    )


def load_widget_frame(frame_value):
    path = resolve_frame_path_for_widget(frame_value)

    if path is None:
        img = Image.new("RGB", (1280, 720), color=(30, 30, 30))
        return img, None

    img = Image.open(path).convert("RGB")
    return img, path


def draw_box(draw, bbox, outline, width=3):
    if bbox is None:
        return

    try:
        x1, y1, x2, y2 = [float(v) for v in bbox]
    except Exception:
        return

    if not np.isfinite([x1, y1, x2, y2]).all():
        return

    draw.rectangle([x1, y1, x2, y2], outline=outline, width=width)


def get_frame_row(frame_value):
    subset = df_solver[df_solver["Frame"] == frame_value]

    if len(subset) == 0:
        return None

    return subset.iloc[0]


def annotate_solver_frame(
    frame_value,
    show_face_boxes=True,
    show_body_boxes=True,
    show_identity=True,
):
    img, frame_path = load_widget_frame(frame_value)
    draw = ImageDraw.Draw(img)

    frame_row = get_frame_row(frame_value)
    frame_dets = out[out["frame"] == frame_value].copy()
    frame_dets = frame_dets.sort_values("face_cx")

    # Blue = cleaned body boxes
    if show_body_boxes and frame_row is not None:
        poses = frame_row["Poses"] if isinstance(frame_row["Poses"], list) else []

        for i, pose in enumerate(poses):
            if not isinstance(pose, dict) or "bbox" not in pose:
                continue

            bbox = pose["bbox"]

            draw_box(draw, bbox, outline=(0, 150, 255), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]

            draw_text(
                draw,
                x1,
                max(0, y1 - 22),
                f"body_{i}",
                fill=(0, 180, 255),
                font_size=13,
            )

    # Green = cleaned face boxes
    if show_face_boxes and frame_row is not None:
        faces = frame_row["Fer"] if isinstance(frame_row["Fer"], list) else []

        for i, face in enumerate(faces):
            if not isinstance(face, dict) or "bbox" not in face:
                continue

            bbox = face["bbox"]

            draw_box(draw, bbox, outline=(0, 255, 0), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]

            draw_text(
                draw,
                x1,
                y2 + 4,
                f"face_{i}",
                fill=(0, 255, 0),
                font_size=13,
            )

    # Yellow = final identity around solver face detection
    if show_identity:
        for _, r in frame_dets.iterrows():
            bbox = [r["face_x1"], r["face_y1"], r["face_x2"], r["face_y2"]]

            label = str(r["person_label"])
            conf = float(r["confidence"])

            label_text = f"{label} {conf:.2f}"

            draw_box(draw, bbox, outline=(255, 255, 0), width=5)

            x1, y1, x2, y2 = bbox

            draw_text(
                draw,
                x1,
                max(0, y1 - 46),
                label_text,
                fill=(255, 255, 0),
                font_size=20,
            )

    if len(frame_dets) > 0:
        first = frame_dets.iloc[0]
        info = (
            f"frame={first['frame_num']} | "
            f"faces={first['n_faces']} | "
            f"poses={first['n_poses']} | "
            f"source={first['assignment_source']}"
        )
    elif frame_row is not None:
        info = (
            f"frame={frame_row['frame_num']} | "
            f"faces={frame_row['n_faces']} | "
            f"poses={frame_row['n_poses']} | "
            f"No solver detections"
        )
    else:
        info = "No detections"

    draw_text(
        draw,
        10,
        img.size[1] - 35,
        info,
        fill=(255, 255, 255),
        font_size=15,
    )

    return img, frame_path

In [ ]:
frame_table = (
    df_solver[["Frame", "frame_num", "n_faces", "n_poses"]]
    .drop_duplicates()
    .sort_values("frame_num")
    .reset_index(drop=True)
)

frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_table) - 1,
    step=1,
    description="Frame",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)

show_body_checkbox = widgets.Checkbox(
    value=True,
    description="Body boxes",
)

show_face_checkbox = widgets.Checkbox(
    value=True,
    description="Face boxes",
)

show_identity_checkbox = widgets.Checkbox(
    value=True,
    description="Identity",
)

jump_frame = widgets.IntText(
    value=0,
    description="Go to frame:",
)

jump_button = widgets.Button(
    description="Jump",
    button_style="info",
)

output = widgets.Output()


def jump_to_frame(_):
    target = int(jump_frame.value)

    idx = (frame_table["frame_num"] - target).abs().idxmin()

    frame_slider.value = int(idx)


jump_button.on_click(jump_to_frame)


def update_frame(change=None):
    idx = frame_slider.value
    row = frame_table.iloc[idx]

    frame_value = row["Frame"]

    with output:
        clear_output(wait=True)

        img, frame_path = annotate_solver_frame(
            frame_value,
            show_face_boxes=show_face_checkbox.value,
            show_body_boxes=show_body_checkbox.value,
            show_identity=show_identity_checkbox.value,
        )

        plt.figure(figsize=(13, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.show()

        frame_dets = out[out["frame"] == frame_value].sort_values("face_cx")

        print("slider index:", idx)
        print("frame_num:", row["frame_num"])
        print("frame:", frame_value)
        print("frame path:", frame_path)
        print("cleaned faces:", row["n_faces"])
        print("cleaned bodies:", row["n_poses"])
        print("solver detections in frame:", len(frame_dets))

        display_cols = [
            "face_idx_lr",
            "person_label",
            "confidence",
            "assignment_source",
            "model_person",
            "model_confidence",
            "unconstrained_model_person",
            "unconstrained_model_confidence",
            "weak_source",
        ]

        display_cols = [c for c in display_cols if c in frame_dets.columns]

        if len(frame_dets) > 0:
            display(frame_dets[display_cols])


frame_slider.observe(update_frame, names="value")
show_body_checkbox.observe(update_frame, names="value")
show_face_checkbox.observe(update_frame, names="value")
show_identity_checkbox.observe(update_frame, names="value")

display(
    widgets.VBox(
        [
            frame_slider,
            widgets.HBox(
                [
                    show_body_checkbox,
                    show_face_checkbox,
                    show_identity_checkbox,
                ]
            ),
            widgets.HBox([jump_frame, jump_button]),
            output,
        ]
    )
)

update_frame()

In [ ]:
# ============================================================
# NEW CELL: Infer real names for person_A / person_C using voice CSV
# ============================================================

from itertools import permutations

SPEAKER_CSV = DATA_DIR / "speaker_by_second_all_debates_named_clean.csv"

# Name used inside the speaker CSV
VOICE_DEBATE_NAME = "Ventura vs Seguro"

MIN_SINGLE_RUN = 10          # 10 consecutive single-person frames
MIN_DOMINANT_SHARE = 0.60    # speaker must dominate at least 60% of that run
MIN_MEAN_CONFIDENCE = 0.55   # visual identity confidence threshold
VOICE_TIME_OFFSET = 0        # change only if audio seconds and frame_num are shifted

if not SPEAKER_CSV.exists():
    print("Could not find speaker CSV at:", SPEAKER_CSV)
    print("CSV files found in data folder:")
    for p in DATA_DIR.rglob("*.csv"):
        print(" -", p)
    raise FileNotFoundError(SPEAKER_CSV)

voice_all = pd.read_csv(SPEAKER_CSV)

voice_df = (
    voice_all[voice_all["debate_name"] == VOICE_DEBATE_NAME]
    .copy()
    .sort_values("second")
    .reset_index(drop=True)
)

if len(voice_df) == 0:
    print("Available debate names:")
    display(voice_all["debate_name"].drop_duplicates().sort_values())
    raise ValueError(f"No rows found for debate_name = {VOICE_DEBATE_NAME}")

print("Loaded voice rows:", len(voice_df))
print("Speakers in CSV:")
display(voice_df["speaker_name"].value_counts())

candidate_names = sorted(
    set(voice_df["candidate_1"].dropna().unique()).union(
        set(voice_df["candidate_2"].dropna().unique())
    )
)

print("Candidate names:", candidate_names)

# ------------------------------------------------------------
# Single-person visual frames
# ------------------------------------------------------------
single_frames = (
    out.loc[
        out["n_faces"] == 1,
        ["frame", "frame_num", "person_label", "confidence"]
    ]
    .drop_duplicates(subset=["frame_num"])
    .sort_values("frame_num")
    .reset_index(drop=True)
)

single_frames["voice_second"] = single_frames["frame_num"].astype(int) + VOICE_TIME_OFFSET

single_frames = single_frames.merge(
    voice_df[["second", "speaker_name", "voice_label", "estimated_speaker"]],
    left_on="voice_second",
    right_on="second",
    how="left",
)

# ------------------------------------------------------------
# Consecutive runs of the SAME visual person
# ------------------------------------------------------------
single_frames["new_run"] = (
    single_frames["person_label"].ne(single_frames["person_label"].shift())
    | single_frames["frame_num"].diff().fillna(1).ne(1)
)

single_frames["run_id"] = single_frames["new_run"].cumsum()

run_rows = []

for run_id, g in single_frames.groupby("run_id"):
    g = g.sort_values("frame_num")

    visual_label = g["person_label"].iloc[0]
    run_length = len(g)

    # We only infer candidates. person_B is fixed as moderator/other.
    if visual_label not in ["person_A", "person_C"]:
        continue

    if run_length < MIN_SINGLE_RUN:
        continue

    start_sec = int(g["frame_num"].min())
    end_sec = int(g["frame_num"].max())
    mean_confidence = float(g["confidence"].mean())

    candidate_speaking = g[g["speaker_name"].isin(candidate_names)]

    if len(candidate_speaking) == 0:
        dominant_speaker = None
        dominant_count = 0
        dominant_share = 0.0
    else:
        counts = candidate_speaking["speaker_name"].value_counts()
        dominant_speaker = counts.index[0]
        dominant_count = int(counts.iloc[0])
        dominant_share = dominant_count / run_length

    run_rows.append(
        {
            "run_id": int(run_id),
            "person_label": visual_label,
            "start_sec": start_sec,
            "end_sec": end_sec,
            "run_length": run_length,
            "mean_confidence": mean_confidence,
            "dominant_speaker": dominant_speaker,
            "dominant_count": dominant_count,
            "dominant_share": dominant_share,
        }
    )

named_run_summary = pd.DataFrame(run_rows)

print("All candidate single-person runs:")
display(named_run_summary.sort_values(["person_label", "start_sec"]).head(30))

# ------------------------------------------------------------
# Reliable runs only
# ------------------------------------------------------------
reliable_named_runs = named_run_summary[
    (named_run_summary["run_length"] >= MIN_SINGLE_RUN)
    & (named_run_summary["dominant_share"] >= MIN_DOMINANT_SHARE)
    & (named_run_summary["mean_confidence"] >= MIN_MEAN_CONFIDENCE)
    & (named_run_summary["dominant_speaker"].notna())
].copy()

print("Reliable runs used for mapping:", len(reliable_named_runs))
display(
    reliable_named_runs
    .sort_values(["person_label", "run_length"], ascending=[True, False])
    .head(30)
)

# ------------------------------------------------------------
# Voting table: visual person -> spoken name
# ------------------------------------------------------------
if len(reliable_named_runs) > 0:
    named_mapping_votes = (
        reliable_named_runs
        .groupby(["person_label", "dominant_speaker"])
        .agg(
            n_runs=("run_id", "count"),
            total_seconds=("run_length", "sum"),
            mean_run_length=("run_length", "mean"),
            mean_share=("dominant_share", "mean"),
            mean_confidence=("mean_confidence", "mean"),
        )
        .reset_index()
        .sort_values(
            ["person_label", "total_seconds", "n_runs", "mean_share", "mean_confidence"],
            ascending=[True, False, False, False, False],
        )
    )
else:
    named_mapping_votes = pd.DataFrame(
        columns=[
            "person_label",
            "dominant_speaker",
            "n_runs",
            "total_seconds",
            "mean_run_length",
            "mean_share",
            "mean_confidence",
        ]
    )

print("Mapping votes:")
display(named_mapping_votes)

# ------------------------------------------------------------
# Choose best assignment for person_A and person_C
# Avoid assigning the same candidate to both when possible.
# ------------------------------------------------------------
score_table = {
    (row["person_label"], row["dominant_speaker"]): float(row["total_seconds"])
    for _, row in named_mapping_votes.iterrows()
}

visual_candidates = ["person_A", "person_C"]

best_score = -1
best_assignment = {}

if len(candidate_names) >= 2:
    for perm in permutations(candidate_names, 2):
        candidate_assignment = {
            "person_A": perm[0],
            "person_C": perm[1],
        }

        score = sum(
            score_table.get((visual_label, speaker_name), 0)
            for visual_label, speaker_name in candidate_assignment.items()
        )

        if score > best_score:
            best_score = score
            best_assignment = candidate_assignment

# Fallback if there was not enough evidence
visual_to_name_map = {
    "person_A": best_assignment.get("person_A", "person_A"),
    "person_B": "Moderador/Other",
    "person_C": best_assignment.get("person_C", "person_C"),
}

print("Final visual label -> display name map:")
display(pd.DataFrame(
    list(visual_to_name_map.items()),
    columns=["visual_label", "display_name"]
))

# Create a separate named dataframe, do not overwrite your original out
out_named = out.copy()
out_named["display_label"] = (
    out_named["person_label"]
    .map(visual_to_name_map)
    .fillna(out_named["person_label"])
)

# Voice lookup for the widget
voice_lookup = voice_df.set_index("second")["speaker_name"].to_dict()

print("Created out_named with real display labels.")

In [ ]:
# ============================================================
# NEW CELL: Second widget, same visuals but with real names
# ============================================================

def annotate_solver_frame_with_names(
    frame_value,
    show_face_boxes=True,
    show_body_boxes=True,
    show_identity=True,
):
    img, frame_path = load_widget_frame(frame_value)
    draw = ImageDraw.Draw(img)

    frame_row = get_frame_row(frame_value)

    frame_dets = out_named[out_named["frame"] == frame_value].copy()
    frame_dets = frame_dets.sort_values("face_cx")

    # Blue = cleaned body boxes
    if show_body_boxes and frame_row is not None:
        poses = frame_row["Poses"] if isinstance(frame_row["Poses"], list) else []

        for i, pose in enumerate(poses):
            if not isinstance(pose, dict) or "bbox" not in pose:
                continue

            bbox = pose["bbox"]
            draw_box(draw, bbox, outline=(0, 150, 255), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]

            draw_text(
                draw,
                x1,
                max(0, y1 - 22),
                f"body_{i}",
                fill=(0, 180, 255),
                font_size=13,
            )

    # Green = cleaned face boxes
    if show_face_boxes and frame_row is not None:
        faces = frame_row["Fer"] if isinstance(frame_row["Fer"], list) else []

        for i, face in enumerate(faces):
            if not isinstance(face, dict) or "bbox" not in face:
                continue

            bbox = face["bbox"]
            draw_box(draw, bbox, outline=(0, 255, 0), width=3)

            x1, y1, x2, y2 = [float(v) for v in bbox]

            draw_text(
                draw,
                x1,
                y2 + 4,
                f"face_{i}",
                fill=(0, 255, 0),
                font_size=13,
            )

    # Yellow = real display name
    if show_identity:
        for _, r in frame_dets.iterrows():
            bbox = [r["face_x1"], r["face_y1"], r["face_x2"], r["face_y2"]]

            visual_label = str(r["person_label"])
            display_label = str(r["display_label"])
            conf = float(r["confidence"])

            label_text = f"{display_label} ({visual_label}) {conf:.2f}"

            draw_box(draw, bbox, outline=(255, 255, 0), width=5)

            x1, y1, x2, y2 = bbox

            draw_text(
                draw,
                x1,
                max(0, y1 - 46),
                label_text,
                fill=(255, 255, 0),
                font_size=20,
            )

    # Bottom information bar
    if len(frame_dets) > 0:
        first = frame_dets.iloc[0]
        frame_num = int(first["frame_num"])
        current_speaker = voice_lookup.get(frame_num + VOICE_TIME_OFFSET, "Unknown")

        info = (
            f"frame={frame_num} | "
            f"faces={first['n_faces']} | "
            f"poses={first['n_poses']} | "
            f"voice_csv={current_speaker} | "
            f"source={first['assignment_source']}"
        )

    elif frame_row is not None:
        frame_num = int(frame_row["frame_num"])
        current_speaker = voice_lookup.get(frame_num + VOICE_TIME_OFFSET, "Unknown")

        info = (
            f"frame={frame_num} | "
            f"faces={frame_row['n_faces']} | "
            f"poses={frame_row['n_poses']} | "
            f"voice_csv={current_speaker} | "
            f"No solver detections"
        )

    else:
        info = "No detections"

    draw_text(
        draw,
        10,
        img.size[1] - 35,
        info,
        fill=(255, 255, 255),
        font_size=15,
    )

    return img, frame_path


# ------------------------------------------------------------
# New independent widget controls
# ------------------------------------------------------------
named_frame_table = (
    df_solver[["Frame", "frame_num", "n_faces", "n_poses"]]
    .drop_duplicates()
    .sort_values("frame_num")
    .reset_index(drop=True)
)

named_frame_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(named_frame_table) - 1,
    step=1,
    description="Frame",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)

named_show_body_checkbox = widgets.Checkbox(
    value=True,
    description="Body boxes",
)

named_show_face_checkbox = widgets.Checkbox(
    value=True,
    description="Face boxes",
)

named_show_identity_checkbox = widgets.Checkbox(
    value=True,
    description="Names",
)

named_jump_frame = widgets.IntText(
    value=0,
    description="Go to frame:",
)

named_jump_button = widgets.Button(
    description="Jump",
    button_style="success",
)

named_output = widgets.Output()

named_mapping_html = widgets.HTML(
    value=(
        "<h3>Named visual identity widget</h3>"
        "<p>Automatically inferred mapping from single-person speaking runs:</p>"
        + pd.DataFrame(
            list(visual_to_name_map.items()),
            columns=["visual_label", "display_name"]
        ).to_html(index=False)
    )
)


def named_jump_to_frame(_):
    target = int(named_jump_frame.value)
    idx = (named_frame_table["frame_num"] - target).abs().idxmin()
    named_frame_slider.value = int(idx)


named_jump_button.on_click(named_jump_to_frame)


def update_named_frame(change=None):
    idx = named_frame_slider.value
    row = named_frame_table.iloc[idx]

    frame_value = row["Frame"]
    frame_num = int(row["frame_num"])

    with named_output:
        clear_output(wait=True)

        img, frame_path = annotate_solver_frame_with_names(
            frame_value,
            show_face_boxes=named_show_face_checkbox.value,
            show_body_boxes=named_show_body_checkbox.value,
            show_identity=named_show_identity_checkbox.value,
        )

        plt.figure(figsize=(13, 8))
        plt.imshow(img)
        plt.axis("off")
        plt.show()

        frame_dets = out_named[out_named["frame"] == frame_value].sort_values("face_cx")
        current_speaker = voice_lookup.get(frame_num + VOICE_TIME_OFFSET, "Unknown")

        print("slider index:", idx)
        print("frame_num:", frame_num)
        print("frame:", frame_value)
        print("frame path:", frame_path)
        print("cleaned faces:", row["n_faces"])
        print("cleaned bodies:", row["n_poses"])
        print("speaker from CSV at this second:", current_speaker)
        print("solver detections in frame:", len(frame_dets))

        display_cols = [
            "face_idx_lr",
            "person_label",
            "display_label",
            "confidence",
            "assignment_source",
            "model_person",
            "model_confidence",
            "unconstrained_model_person",
            "unconstrained_model_confidence",
            "weak_source",
        ]

        display_cols = [c for c in display_cols if c in frame_dets.columns]

        if len(frame_dets) > 0:
            display(frame_dets[display_cols])


named_frame_slider.observe(update_named_frame, names="value")
named_show_body_checkbox.observe(update_named_frame, names="value")
named_show_face_checkbox.observe(update_named_frame, names="value")
named_show_identity_checkbox.observe(update_named_frame, names="value")

display(
    widgets.VBox(
        [
            named_mapping_html,
            named_frame_slider,
            widgets.HBox(
                [
                    named_show_body_checkbox,
                    named_show_face_checkbox,
                    named_show_identity_checkbox,
                ]
            ),
            widgets.HBox([named_jump_frame, named_jump_button]),
            named_output,
        ]
    )
)

update_named_frame()

In [ ]:
# ============================================================
# VISUAL -> AUDIO EXPORT
# Creates visual timelines that audio colleague can merge with audio features
# ============================================================

import json
import numpy as np
import pandas as pd

# Use inferred frame size if it exists, otherwise fallback to assignment default
FRAME_WIDTH = globals().get("width", 1280)
FRAME_HEIGHT = globals().get("height", 720)

WINDOW_SIZE = 10  # seconds

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def safe_emotion_confidence(row):
    """
    Extract confidence of the top emotion from emotion_probs_json.
    Returns NaN if not available.
    """
    top_emotion = row.get("top_emotion", None)

    if top_emotion is None or pd.isna(top_emotion) or top_emotion == "":
        return np.nan

    probs_raw = row.get("emotion_probs_json", None)

    if probs_raw is None or pd.isna(probs_raw) or probs_raw == "":
        return np.nan

    try:
        probs = json.loads(probs_raw)
        return float(probs.get(top_emotion, np.nan))
    except Exception:
        return np.nan


def mode_or_none(series):
    series = series.dropna()
    series = series[series.astype(str) != ""]

    if len(series) == 0:
        return None

    return series.value_counts().index[0]


def keypoint_movement(current_kp, previous_kp, indices=None):
    """
    Computes average keypoint movement between consecutive frames.
    Coordinates are normalized by frame width/height.
    """
    try:
        cur = np.asarray(current_kp, dtype=float)
        prev = np.asarray(previous_kp, dtype=float)
    except Exception:
        return np.nan

    if cur.ndim != 2 or prev.ndim != 2:
        return np.nan

    if cur.shape[0] == 0 or prev.shape[0] == 0:
        return np.nan

    n = min(cur.shape[0], prev.shape[0])

    cur = cur[:n]
    prev = prev[:n]

    if indices is not None:
        valid_indices = [i for i in indices if i < n]
        if len(valid_indices) == 0:
            return np.nan
        cur = cur[valid_indices]
        prev = prev[valid_indices]

    # x/y coordinates
    cur_xy = cur[:, :2].astype(float)
    prev_xy = prev[:, :2].astype(float)

    # Optional confidence column
    if cur.shape[1] >= 3 and prev.shape[1] >= 3:
        conf_mask = (cur[:, 2] > 0.20) & (prev[:, 2] > 0.20)
    else:
        conf_mask = np.ones(len(cur_xy), dtype=bool)

    finite_mask = (
        np.isfinite(cur_xy).all(axis=1)
        & np.isfinite(prev_xy).all(axis=1)
        & conf_mask
    )

    if finite_mask.sum() == 0:
        return np.nan

    cur_norm = cur_xy[finite_mask] / np.array([FRAME_WIDTH, FRAME_HEIGHT])
    prev_norm = prev_xy[finite_mask] / np.array([FRAME_WIDTH, FRAME_HEIGHT])

    distances = np.sqrt(((cur_norm - prev_norm) ** 2).sum(axis=1))

    return float(np.nanmean(distances))


# ------------------------------------------------------------
# Build per-second visual timeline
# ------------------------------------------------------------

visual_timeline = out_named.copy()

visual_timeline["debate_name"] = DEBATE_NAME
visual_timeline["second"] = visual_timeline["frame_num"].astype(int)

# Make sure display label exists
if "display_label" not in visual_timeline.columns:
    visual_timeline["display_label"] = visual_timeline["person_label"]

# Face geometry
visual_timeline["face_cx_norm"] = visual_timeline["face_cx"] / FRAME_WIDTH
visual_timeline["face_cy_norm"] = visual_timeline["face_cy"] / FRAME_HEIGHT

visual_timeline["face_width_norm"] = (
    visual_timeline["face_x2"] - visual_timeline["face_x1"]
) / FRAME_WIDTH

visual_timeline["face_height_norm"] = (
    visual_timeline["face_y2"] - visual_timeline["face_y1"]
) / FRAME_HEIGHT

# Emotion confidence
if "emotion_probs_json" in visual_timeline.columns:
    visual_timeline["top_emotion_confidence"] = visual_timeline.apply(
        safe_emotion_confidence,
        axis=1,
    )
else:
    visual_timeline["top_emotion_confidence"] = np.nan

visual_timeline["top_emotion_clean"] = (
    visual_timeline["top_emotion"]
    .fillna("unknown")
    .astype(str)
)

visual_timeline["is_non_neutral_emotion"] = (
    ~visual_timeline["top_emotion_clean"].str.lower().isin(
        ["neutral", "unknown", "none", ""]
    )
).astype(int)

visual_timeline["non_neutral_emotion_score"] = (
    visual_timeline["is_non_neutral_emotion"]
    * visual_timeline["top_emotion_confidence"].fillna(0)
)

# ------------------------------------------------------------
# Face movement between consecutive frames of same visual person
# ------------------------------------------------------------

visual_timeline = visual_timeline.sort_values(
    ["display_label", "second", "face_idx_lr"]
).reset_index(drop=True)

visual_timeline["prev_second"] = visual_timeline.groupby("display_label")["second"].shift(1)
visual_timeline["prev_face_cx_norm"] = visual_timeline.groupby("display_label")["face_cx_norm"].shift(1)
visual_timeline["prev_face_cy_norm"] = visual_timeline.groupby("display_label")["face_cy_norm"].shift(1)

visual_timeline["is_consecutive_frame"] = (
    visual_timeline["second"] - visual_timeline["prev_second"] == 1
)

visual_timeline["face_movement"] = np.where(
    visual_timeline["is_consecutive_frame"],
    np.sqrt(
        (visual_timeline["face_cx_norm"] - visual_timeline["prev_face_cx_norm"]) ** 2
        + (visual_timeline["face_cy_norm"] - visual_timeline["prev_face_cy_norm"]) ** 2
    ),
    np.nan,
)

# ------------------------------------------------------------
# Pose movement and hand movement
# ------------------------------------------------------------

visual_timeline["pose_movement"] = np.nan
visual_timeline["hand_movement"] = np.nan

# COCO-like pose indices:
# shoulders: 5, 6
# elbows: 7, 8
# wrists: 9, 10
HAND_AND_ARM_INDICES = [7, 8, 9, 10]

if "pose_keypoints" in visual_timeline.columns:
    for label, group in visual_timeline.groupby("display_label"):
        group = group.sort_values("second")

        previous_idx = None
        previous_kp = None
        previous_second = None

        for idx, row in group.iterrows():
            current_second = int(row["second"])
            current_kp = row["pose_keypoints"]

            if (
                previous_idx is not None
                and previous_second is not None
                and current_second - previous_second == 1
            ):
                visual_timeline.at[idx, "pose_movement"] = keypoint_movement(
                    current_kp,
                    previous_kp,
                    indices=None,
                )

                visual_timeline.at[idx, "hand_movement"] = keypoint_movement(
                    current_kp,
                    previous_kp,
                    indices=HAND_AND_ARM_INDICES,
                )

            previous_idx = idx
            previous_kp = current_kp
            previous_second = current_second

# Final movement score
visual_timeline["movement_score"] = visual_timeline[
    ["face_movement", "pose_movement", "hand_movement"]
].mean(axis=1, skipna=True)

# Percentile movement per person
visual_timeline["movement_percentile_by_person"] = (
    visual_timeline
    .groupby("display_label")["movement_score"]
    .rank(pct=True)
)

visual_timeline["high_movement_flag"] = (
    visual_timeline["movement_percentile_by_person"] >= 0.90
).astype(int)

visual_timeline["strong_emotion_flag"] = (
    visual_timeline["non_neutral_emotion_score"] >= 0.60
).astype(int)

visual_timeline["visual_interest_flag"] = (
    (visual_timeline["high_movement_flag"] == 1)
    | (visual_timeline["strong_emotion_flag"] == 1)
).astype(int)

# ------------------------------------------------------------
# Keep useful columns for audio colleague
# ------------------------------------------------------------

visual_second_cols = [
    "debate_name",
    "second",
    "frame_num",
    "display_label",
    "person_label",
    "confidence",
    "assignment_source",
    "n_faces",
    "n_poses",
    "face_cx_norm",
    "face_cy_norm",
    "face_area_norm",
    "face_width_norm",
    "face_height_norm",
    "top_emotion_clean",
    "top_emotion_confidence",
    "non_neutral_emotion_score",
    "face_movement",
    "pose_movement",
    "hand_movement",
    "movement_score",
    "movement_percentile_by_person",
    "high_movement_flag",
    "strong_emotion_flag",
    "visual_interest_flag",
]

visual_second_cols = [c for c in visual_second_cols if c in visual_timeline.columns]

visual_second_export = visual_timeline[visual_second_cols].copy()

# Save per-second CSV
visual_second_path = PROCESSED_DIR / f"{DEBATE_NAME}_visual_second_timeline_named.csv"
visual_second_export.to_csv(visual_second_path, index=False)

print("Saved per-second visual timeline:")
print(visual_second_path)
display(visual_second_export.head())

In [ ]:
# ============================================================
# VISUAL 10-SECOND WINDOWS
# Better for correlation with speech rate, pitch, etc.
# ============================================================

visual_second_export["window_start"] = (
    visual_second_export["second"] // WINDOW_SIZE
) * WINDOW_SIZE

visual_second_export["window_end"] = (
    visual_second_export["window_start"] + WINDOW_SIZE - 1
)

# Emotion share table per person/window
emotion_counts = (
    visual_second_export
    .groupby(["debate_name", "display_label", "window_start", "top_emotion_clean"])
    .size()
    .reset_index(name="emotion_count")
)

emotion_totals = (
    visual_second_export
    .groupby(["debate_name", "display_label", "window_start"])
    .size()
    .reset_index(name="visible_seconds")
)

emotion_counts = emotion_counts.merge(
    emotion_totals,
    on=["debate_name", "display_label", "window_start"],
    how="left",
)

emotion_counts["emotion_share"] = (
    emotion_counts["emotion_count"] / emotion_counts["visible_seconds"]
)

emotion_pivot = emotion_counts.pivot_table(
    index=["debate_name", "display_label", "window_start"],
    columns="top_emotion_clean",
    values="emotion_share",
    fill_value=0,
).reset_index()

emotion_pivot.columns = [
    f"emotion_share_{c}" if c not in ["debate_name", "display_label", "window_start"] else c
    for c in emotion_pivot.columns
]

# Main window summary
visual_windows = (
    visual_second_export
    .groupby(["debate_name", "display_label", "window_start"])
    .agg(
        window_end=("window_end", "max"),
        visible_seconds=("second", "count"),
        mean_identity_confidence=("confidence", "mean"),

        mean_movement=("movement_score", "mean"),
        max_movement=("movement_score", "max"),
        mean_face_movement=("face_movement", "mean"),
        mean_pose_movement=("pose_movement", "mean"),
        mean_hand_movement=("hand_movement", "mean"),

        dominant_emotion=("top_emotion_clean", mode_or_none),
        mean_emotion_confidence=("top_emotion_confidence", "mean"),
        mean_non_neutral_emotion_score=("non_neutral_emotion_score", "mean"),

        high_movement_seconds=("high_movement_flag", "sum"),
        strong_emotion_seconds=("strong_emotion_flag", "sum"),
        visually_interesting_seconds=("visual_interest_flag", "sum"),
    )
    .reset_index()
)

visual_windows = visual_windows.merge(
    emotion_pivot,
    on=["debate_name", "display_label", "window_start"],
    how="left",
)

# Require at least a few visible seconds in the window
visual_windows["visible_share_of_window"] = (
    visual_windows["visible_seconds"] / WINDOW_SIZE
)

visual_windows["high_movement_share"] = (
    visual_windows["high_movement_seconds"] / visual_windows["visible_seconds"]
)

visual_windows["strong_emotion_share"] = (
    visual_windows["strong_emotion_seconds"] / visual_windows["visible_seconds"]
)

visual_windows["visual_interest_share"] = (
    visual_windows["visually_interesting_seconds"] / visual_windows["visible_seconds"]
)

# Simple ranking score for colleague to inspect first
visual_windows["visual_intensity_score"] = (
    visual_windows["mean_movement"].rank(pct=True)
    + visual_windows["mean_non_neutral_emotion_score"].rank(pct=True)
    + visual_windows["mean_hand_movement"].rank(pct=True)
)

visual_windows = visual_windows.sort_values(
    ["window_start", "display_label"]
).reset_index(drop=True)

visual_windows_path = PROCESSED_DIR / f"{DEBATE_NAME}_visual_10s_windows_named.csv"
visual_windows.to_csv(visual_windows_path, index=False)

print("Saved 10-second visual window file:")
print(visual_windows_path)

display(visual_windows.head(20))

In [ ]:
# ============================================================
# VISUAL EVENTS FOR AUDIO COLLEAGUE TO CHECK
# ============================================================

# Ignore moderator if you want candidate-focused analysis
candidate_visual_events = visual_windows[
    visual_windows["display_label"] != "Moderador/Other"
].copy()

# Keep windows where the person is visible for at least half the window
candidate_visual_events = candidate_visual_events[
    candidate_visual_events["visible_share_of_window"] >= 0.50
].copy()

# Top visually intense windows
visual_events_for_audio = (
    candidate_visual_events
    .sort_values("visual_intensity_score", ascending=False)
    .head(40)
    .copy()
)

visual_events_for_audio = visual_events_for_audio[
    [
        "debate_name",
        "display_label",
        "window_start",
        "window_end",
        "visible_seconds",
        "dominant_emotion",
        "mean_movement",
        "max_movement",
        "mean_hand_movement",
        "mean_non_neutral_emotion_score",
        "high_movement_share",
        "strong_emotion_share",
        "visual_interest_share",
        "visual_intensity_score",
    ]
]

visual_events_path = PROCESSED_DIR / f"{DEBATE_NAME}_visual_events_for_audio_check.csv"
visual_events_for_audio.to_csv(visual_events_path, index=False)

print("Saved visual events for audio colleague:")
print(visual_events_path)

display(visual_events_for_audio)

In [ ]:
# ============================================================
# VISUAL EXPORT SANITY CHECKS
# ============================================================

print("visual_second_export shape:", visual_second_export.shape)
print("visual_windows shape:", visual_windows.shape)

print("\nPeople detected:")
display(visual_second_export["display_label"].value_counts())

print("\nEmotion distribution:")
display(visual_second_export["top_emotion_clean"].value_counts())

print("\nMissing values:")
display(
    visual_second_export[
        [
            "movement_score",
            "face_movement",
            "pose_movement",
            "hand_movement",
            "top_emotion_confidence",
        ]
    ].isna().mean().sort_values(ascending=False)
)

print("\nMovement summary by person:")
display(
    visual_second_export
    .groupby("display_label")
    .agg(
        visible_seconds=("second", "count"),
        mean_movement=("movement_score", "mean"),
        max_movement=("movement_score", "max"),
        mean_hand_movement=("hand_movement", "mean"),
        high_movement_seconds=("high_movement_flag", "sum"),
        strong_emotion_seconds=("strong_emotion_flag", "sum"),
    )
    .sort_values("mean_movement", ascending=False)
)

print("\nTop visual event windows:")
display(
    visual_windows
    .sort_values("visual_intensity_score", ascending=False)
    .head(15)
    [
        [
            "display_label",
            "window_start",
            "window_end",
            "visible_seconds",
            "dominant_emotion",
            "mean_movement",
            "mean_hand_movement",
            "mean_non_neutral_emotion_score",
            "visual_intensity_score",
        ]
    ]
)

In [ ]:
# ============================================================
# VISUAL MOVEMENT TIMELINE
# ============================================================

plt.figure(figsize=(15, 5))

for person_name, g in visual_second_export.groupby("display_label"):
    g = g.sort_values("second")

    # Smooth movement a bit so the plot is readable
    y = g["movement_score"].rolling(5, min_periods=1).mean()

    plt.plot(
        g["second"],
        y,
        label=person_name,
        linewidth=1.5,
    )

plt.title("Visual movement over time")
plt.xlabel("Second")
plt.ylabel("Movement score, rolling mean")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# HAND / ARM MOVEMENT TIMELINE
# ============================================================

plt.figure(figsize=(15, 5))

for person_name, g in visual_second_export.groupby("display_label"):
    g = g.sort_values("second")

    y = g["hand_movement"].rolling(5, min_periods=1).mean()

    plt.plot(
        g["second"],
        y,
        label=person_name,
        linewidth=1.5,
    )

plt.title("Hand/arm movement over time")
plt.xlabel("Second")
plt.ylabel("Hand movement score, rolling mean")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# VISUAL INTENSITY BY 10-SECOND WINDOW
# ============================================================

plt.figure(figsize=(15, 5))

for person_name, g in visual_windows.groupby("display_label"):
    g = g.sort_values("window_start")

    plt.plot(
        g["window_start"],
        g["visual_intensity_score"],
        marker="o",
        markersize=3,
        label=person_name,
    )

plt.title("Visual intensity score by 10-second window")
plt.xlabel("Window start second")
plt.ylabel("Visual intensity score")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# WIDGET: INSPECT TOP VISUAL EVENTS
# ============================================================

top_events_to_check = (
    visual_windows
    .sort_values("visual_intensity_score", ascending=False)
    .reset_index(drop=True)
    .copy()
)

event_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=min(len(top_events_to_check) - 1, 50),
    step=1,
    description="Event",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)

event_output = widgets.Output()


def show_event(change=None):
    event_idx = int(event_slider.value)
    event = top_events_to_check.iloc[event_idx]

    start_sec = int(event["window_start"])
    end_sec = int(event["window_end"])
    mid_sec = int((start_sec + end_sec) / 2)

    person = event["display_label"]

    seconds_to_show = [start_sec, mid_sec, end_sec]

    with event_output:
        clear_output(wait=True)

        print("Event index:", event_idx)
        print("Person:", person)
        print("Window:", start_sec, "-", end_sec)
        print("Dominant emotion:", event["dominant_emotion"])
        print("Mean movement:", event["mean_movement"])
        print("Mean hand movement:", event["mean_hand_movement"])
        print("Non-neutral emotion score:", event["mean_non_neutral_emotion_score"])
        print("Visual intensity score:", event["visual_intensity_score"])

        display(pd.DataFrame([event]))

        for sec in seconds_to_show:
            idx = (frame_table["frame_num"] - sec).abs().idxmin()
            frame_value = frame_table.iloc[idx]["Frame"]

            img, frame_path = annotate_solver_frame_with_names(
                frame_value,
                show_face_boxes=True,
                show_body_boxes=True,
                show_identity=True,
            )

            plt.figure(figsize=(13, 8))
            plt.imshow(img)
            plt.axis("off")
            plt.title(f"Second {sec} | {person}")
            plt.show()


event_slider.observe(show_event, names="value")

display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Inspect top visual intensity events</h3>"),
            event_slider,
            event_output,
        ]
    )
)

show_event()